In [1]:
!pip install pyspark -q
!pip install boto3 -q
!pip install pyarrow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

Mounted at /content/drive
✅ Drive mounted successfully


In [3]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [4]:
import boto3
import os
import glob
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# ── boto3 Client ──────────────────────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    endpoint_url=G_MINIO_ENDPOINT,
    aws_access_key_id=G_MINIO_ACCESS_KEY,
    aws_secret_access_key=G_MINIO_SECRET_KEY
)

# ── Download All Partitioned Files from MinIO ─────────────────────────────────
local_dir  = "/tmp/earthquake"
bucket     = "rawdatasets"
prefix     = "earthquake/"

os.makedirs(local_dir, exist_ok=True)

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

for obj in response.get("Contents", []):
    s3_key = obj["Key"]

    # Skip if the key is just the folder prefix itself
    if s3_key.endswith("/"):
        continue

    # Reconstruct local path preserving year=/month= structure
    relative_path = os.path.relpath(s3_key, prefix)
    local_file_path = os.path.join(local_dir, relative_path)

    # Create subdirectories if needed (e.g. year=2023/month=6/)
    os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

    s3.download_file(Bucket=bucket, Key=s3_key, Filename=local_file_path)
    print(f"Downloaded: {s3_key} → {local_file_path}")

# ── Read All Partitions into Spark DataFrame ──────────────────────────────────
spark = SparkSession.builder \
    .appName("ReadEarthquakeParquet") \
    .getOrCreate()

df = spark.read.parquet(local_dir)

df.printSchema()
df.show(10, truncate=False)

# ── Cleanup ───────────────────────────────────────────────────────────────────
shutil.rmtree(local_dir, ignore_errors=True)
print(f"Deleted temp directory: {local_dir}")

Downloaded: earthquake/year=2023/month=10/part-00000-3260f660-eb10-406f-9555-87a011966afb.c000.snappy.parquet → /tmp/earthquake/year=2023/month=10/part-00000-3260f660-eb10-406f-9555-87a011966afb.c000.snappy.parquet
Downloaded: earthquake/year=2023/month=11/part-00000-3260f660-eb10-406f-9555-87a011966afb.c000.snappy.parquet → /tmp/earthquake/year=2023/month=11/part-00000-3260f660-eb10-406f-9555-87a011966afb.c000.snappy.parquet
Downloaded: earthquake/year=2023/month=12/part-00000-3260f660-eb10-406f-9555-87a011966afb.c000.snappy.parquet → /tmp/earthquake/year=2023/month=12/part-00000-3260f660-eb10-406f-9555-87a011966afb.c000.snappy.parquet
Downloaded: earthquake/year=2024/month=1/part-00000-3260f660-eb10-406f-9555-87a011966afb.c000.snappy.parquet → /tmp/earthquake/year=2024/month=1/part-00000-3260f660-eb10-406f-9555-87a011966afb.c000.snappy.parquet
Downloaded: earthquake/year=2024/month=10/part-00000-3260f660-eb10-406f-9555-87a011966afb.c000.snappy.parquet → /tmp/earthquake/year=2024/mont